In [1]:
from getpass import getpass
from sqlalchemy import URL, create_engine, text

# Ask for the password at run time, so it is never saved in the notebook
mysql_password = getpass("MySQL root password: ")

# Address of the MySQL server running on this Mac
server_url = URL.create(
    "mysql+pymysql",          # MySQL, talked to through the PyMySQL driver
    username="root",
    password=mysql_password,
    host="localhost",
    port=3306,
)
server = create_engine(server_url)

# Create a database (a folder for our tables) if it doesn't exist yet
with server.connect() as conn:
    conn.execute(text("CREATE DATABASE IF NOT EXISTS norway_forecast"))
    print(conn.execute(text("SHOW DATABASES")).scalars().all())


['bronze', 'DataWarehouse', 'gold', 'information_schema', 'mysql', 'norway_forecast', 'og_upstream', 'performance_schema', 'silver', 'sys']


In [2]:
import pandas as pd
from pathlib import Path

# All raw snapshots, sorted by name. The names contain dates as YYYY-MM-DD,
# so alphabetical order is also date order and the last one is the newest.
snapshots = sorted(Path("../data/raw").glob("field_production_monthly_*.csv"))
latest_snapshot = snapshots[-1]

raw = pd.read_csv(latest_snapshot, encoding="utf-8-sig")
print(f"Read {len(raw):,} rows from {latest_snapshot.name}")


Read 28,455 rows from field_production_monthly_2026-10-13.csv


In [3]:
# Connect to our project database (same server as before, plus the database name)
db = create_engine(server_url.set(database="norway_forecast"))

# Write the table exactly as read. "replace" means re-running this cell
# rebuilds the table instead of adding the rows a second time.
raw.to_sql("bronze_field_production_monthly", db, if_exists="replace", index=False)

# Check with SQL that every row arrived
with db.connect() as conn:
    row_count = conn.execute(
        text("SELECT COUNT(*) FROM bronze_field_production_monthly")
    ).scalar()
print(f"MySQL table has {row_count:,} rows")


MySQL table has 28,455 rows


In [4]:
# DESCRIBE lists every column of a table with its type.
# pd.read_sql runs an SQL query and returns the result as a pandas table.
pd.read_sql(text("DESCRIBE bronze_field_production_monthly"), db)


,Field,Type,Null,Key,Default,Extra
0,prfInformationCarrier,text,YES,,None,
1,prfYear,bigint,YES,,None,
2,prfMonth,bigint,YES,,None,
3,prfPrdOilNetMillSm3,double,YES,,None,
4,prfPrdGasNetBillSm3,double,YES,,None,
5,prfPrdNGLNetMillSm3,double,YES,,None,
6,prfPrdCondensateNetMillSm3,double,YES,,None,
7,prfPrdOeNetMillSm3,double,YES,,None,
8,prfPrdProducedWaterInFieldMillSm3,double,YES,,None,
9,prfNpdidInformationCarrier,bigint,YES,,None,
